# Module 3 - Day 4

**Topics**

- Connecting to databases (sqlite3 ) 
- database connectors
- connecting and executing queries
- Pitfalls in reading/writing to databases.

For today’s practice make use of notebook `module3-day4.ipynb` created in your enviroment. Shut down kernel for all previous notebooks (if in runing condition) by right cliking on notbeook on left hand side file browser

## Working with databases

- You will need a python connector for your database
- Connector is different for each type of database. e.g. mysql provides its connector on the website. Sqilite3 connectors comes with python
- How do you connect to database will be different.
- But working with data, executing queries, fetching or adding data to databses these will be same

In [1]:
import sqlite3

In [ ]:
conn = sqlite3.connect("training.db") # if the database file does not exits, it will create it
cur = conn.cursor()
cur.execute("create table person (name varchar(100), email varchar(100));")

In [3]:
cur.execute("insert into person (name, email) values ('alice', 'alice@wonder.land');")
conn.commit()
conn.close()

In [4]:
!ls training.db

training.db


## Access data

In [5]:
conn = sqlite3.connect("training.db") # this open the database 

In [6]:
cur = conn.cursor() # this will work only if conn is open

In [7]:
results = cur.execute("select * from person") # this means select every field from person table

In [8]:
results # it is acually kind of iterator

In [9]:
nums = [1, 2, 3, 4, 5, 6]

In [10]:
rnums = reversed(nums)

In [11]:
rnums

In [12]:
next(rnums)

6

In [13]:
results.fetchall() # usually data fetched from execute query will be a list of tuples

[('alice', 'alice@wonder.land')]

In [14]:
names = cur.execute("select name from person")

In [15]:
names.fetchall()

[('alice',)]

In [16]:
def find_person(conn, email):
    q = f"select * from person where email='{email}'" # this is not a good way
    print(q)
    cur = conn.cursor()
    r = cur.execute(q)
    return r.fetchall()

In [17]:
find_person(conn, "alice@wonder.land")

select * from person where email='alice@wonder.land'


[('alice', 'alice@wonder.land')]

In [18]:
cur = conn.cursor()
r = cur.execute("select * from person where email=?", ("alice@wonder.land", ))

In [19]:
r.fetchall()

[('alice', 'alice@wonder.land')]

In [29]:
def find_person_with_email(conn, email):
    q = f"select * from person where email=?"
    print(q)
    cur = conn.cursor()
    r = cur.execute(q, (email,))
    return r.fetchall()

In [20]:
[1]

[1]

In [21]:
(1) # this is not tuple

1

In [22]:
(1,) # this is tuple

(1,)

In [23]:
t = (1,)

In [24]:
t[0]

1

In [30]:
find_person_with_email(conn, "alice@wonder.land")

select * from person where email=?


[('alice', 'alice@wonder.land')]

In [27]:
def query(conn, querystring, params=None):
    if params is None:
        params = tuple()
    cur = conn.cursor()
    r = cur.execute(querystring, params)
    return r.fetchall()

In [28]:
query(conn, "insert into person (name, email) values (?,?)", ("alex", "alex@nyk.zoo"))

[]

In [31]:
find_person_with_email(conn, "alex@nyk.zoo")

select * from person where email=?


[('alex', 'alex@nyk.zoo')]

In [33]:
persons = {"xyz":"xyz@xyz.com",
          "abc": "abc@abc.com",
          "mno": "mno@mno.com"}

In [34]:
def insert_person(conn, name, email):
    return query(conn, "insert into person (name, email) values (?,?)", (name, email))

In [35]:
for n, e in persons.items():
    insert_person(conn, n, e)

In [37]:
query(conn, "select * from person")

[('alice', 'alice@wonder.land'),
 ('alex', 'alex@nyk.zoo'),
 ('xyz', 'xyz@xyz.com'),
 ('abc', 'abc@abc.com'),
 ('mno', 'mno@mno.com')]

## Pandas and databases

In [38]:
import pandas as pd

In [39]:
csvlink = "https://raw.githubusercontent.com/vikipedia/python-trainings/master/online_course/source/module2/wallet.csv"
wallet = pd.read_csv(csvlink)

In [40]:
wallet

,Unnamed: 0,date,category,description,debit
0,0,2021-03-07 14:53:28.377359,Music,Amazon,421.207327
1,1,2020-10-08 09:53:28.377359,Food,Swiggy,328.440080
2,2,2021-02-23 09:53:28.377359,Books,Amazon,244.679437
3,3,2020-11-01 14:53:28.377359,Utility,Phone,222.756318
4,4,2021-06-05 13:53:28.377359,Books,Flipcart,494.128492
...,...,...,...,...,...
95,95,2021-07-19 13:53:28.377359,Utility,Phone,388.671213
96,96,2021-01-12 19:53:28.377359,Books,Flipcart,467.554562
97,97,2021-03-25 11:53:28.377359,Utility,Phone,320.789434
98,98,2021-05-13 15:53:28.377359,Travel,Taxi,442.096469


In [41]:
del wallet['Unnamed: 0']

In [42]:
wallet

,date,category,description,debit
0,2021-03-07 14:53:28.377359,Music,Amazon,421.207327
1,2020-10-08 09:53:28.377359,Food,Swiggy,328.440080
2,2021-02-23 09:53:28.377359,Books,Amazon,244.679437
3,2020-11-01 14:53:28.377359,Utility,Phone,222.756318
4,2021-06-05 13:53:28.377359,Books,Flipcart,494.128492
...,...,...,...,...
95,2021-07-19 13:53:28.377359,Utility,Phone,388.671213
96,2021-01-12 19:53:28.377359,Books,Flipcart,467.554562
97,2021-03-25 11:53:28.377359,Utility,Phone,320.789434
98,2021-05-13 15:53:28.377359,Travel,Taxi,442.096469


In [43]:
wallet.to_csv("/tmp/x.csv")

In [44]:
!head /tmp/x.csv

,date,category,description,debit
0,2021-03-07 14:53:28.377359,Music,Amazon,421.2073272347991
1,2020-10-08 09:53:28.377359,Food,Swiggy,328.4400802428426
2,2021-02-23 09:53:28.377359,Books,Amazon,244.67943701511356
3,2020-11-01 14:53:28.377359,Utility,Phone,222.7563175805277
4,2021-06-05 13:53:28.377359,Books,Flipcart,494.1284923793595
5,2021-07-28 19:53:28.377359,Utility,Electricity,219.94171130968408
6,2021-04-16 11:53:28.377359,Books,Amazon Kindle,270.32259514795845
7,2021-02-15 10:53:28.377359,Food,Zomato,457.1831036346536
8,2021-08-10 19:53:28.377359,Utility,Phone,151.49637259947792


In [45]:
wallet.to_sql("exepenses", con=conn, if_exists="replace")

100

In [48]:
r = query(conn, "select * from exepenses")

In [49]:
for item in r:expressions
    print(item)

(0, '2021-03-07 14:53:28.377359', 'Music', 'Amazon', 421.2073272347991)
(1, '2020-10-08 09:53:28.377359', 'Food', 'Swiggy', 328.4400802428426)
(2, '2021-02-23 09:53:28.377359', 'Books', 'Amazon', 244.67943701511356)
(3, '2020-11-01 14:53:28.377359', 'Utility', 'Phone', 222.7563175805277)
(4, '2021-06-05 13:53:28.377359', 'Books', 'Flipcart', 494.1284923793595)
(5, '2021-07-28 19:53:28.377359', 'Utility', 'Electricity', 219.94171130968408)
(6, '2021-04-16 11:53:28.377359', 'Books', 'Amazon Kindle', 270.32259514795845)
(7, '2021-02-15 10:53:28.377359', 'Food', 'Zomato', 457.1831036346536)
(8, '2021-08-10 19:53:28.377359', 'Utility', 'Phone', 151.49637259947792)
(9, '2020-11-29 14:53:28.377359', 'Travel', 'Auto', 443.61888423247854)
(10, '2021-06-15 13:53:28.377359', 'Travel', 'Metro', 328.1754210974373)
(11, '2021-07-24 13:53:28.377359', 'Food', 'Zomato', 434.4954675355444)
(12, '2021-07-24 14:53:28.377359', 'Music', 'Amazon', 329.5360031897569)
(13, '2021-06-06 10:53:28.377359', 'Utilit

In [50]:
def mysum(*nums): # nums is variable number of unamed argumets
    s = 0
    for n in nums:
        s += n
    return s

In [51]:
mysum(1, 2)

3

In [52]:
mysum(1, 2, 3, 4, 5, 6, 7, 7, 8) # these are unnamed arguments

43

In [53]:
def foo(**kwargs): # ** means named arguments
    for name, value in kwargs.items():
        print(name, value)

In [55]:
foo(name="alice", email="alice@wonder.land", place="wanderland") # variable number named arguments

name alice
email alice@wonder.land
place wanderland


In [59]:
def select_query(conn, tablename, *selection, **conditions):
    cur = conn.cursor()
    SELECTION = ",".join(selection)
    CONDITIONS = " & ".join([f"{name} = ?" for name in conditions.keys()])
    params = tuple([conditions[name] for name in conditions.keys()])
    q = f"select {SELECTION} from {tablename} where {CONDITIONS}"
    print(q)
    print(params)
    return cur.execute(q, params)

In [ ]:
select a, b, c, d from tablename where cond1=value and cond2 = value2...

In [57]:
wallet

,date,category,description,debit
0,2021-03-07 14:53:28.377359,Music,Amazon,421.207327
1,2020-10-08 09:53:28.377359,Food,Swiggy,328.440080
2,2021-02-23 09:53:28.377359,Books,Amazon,244.679437
3,2020-11-01 14:53:28.377359,Utility,Phone,222.756318
4,2021-06-05 13:53:28.377359,Books,Flipcart,494.128492
...,...,...,...,...
95,2021-07-19 13:53:28.377359,Utility,Phone,388.671213
96,2021-01-12 19:53:28.377359,Books,Flipcart,467.554562
97,2021-03-25 11:53:28.377359,Utility,Phone,320.789434
98,2021-05-13 15:53:28.377359,Travel,Taxi,442.096469


In [60]:
music = select_query(conn, "exepenses", "description", "debit", category="Music")

select description,debit from exepenses where category = ?
('Music',)


In [62]:
for items in music:
    print(items)

('Amazon', 421.2073272347991)
('Amazon', 329.5360031897569)
('Netflix', 354.9402409919816)
('Amazon', 266.0690783774673)
('spotify', 232.30340219121135)
('spotify', 160.81754340768396)
('Netflix', 188.7487426895118)
('Netflix', 324.786916846731)
('Netflix', 197.5346000167895)
('spotify', 415.3728938035302)
('Netflix', 321.7634156544651)
('spotify', 411.14270120842224)
('Netflix', 158.7936457269333)
('Amazon', 130.37490757527)
('Amazon', 218.487173429263)
('Amazon', 101.57327588889416)


In [65]:
df = pd.read_sql_query("select * from exepenses;", con=conn)

In [66]:
df

,index,date,category,description,debit
0,0,2021-03-07 14:53:28.377359,Music,Amazon,421.207327
1,1,2020-10-08 09:53:28.377359,Food,Swiggy,328.440080
2,2,2021-02-23 09:53:28.377359,Books,Amazon,244.679437
3,3,2020-11-01 14:53:28.377359,Utility,Phone,222.756318
4,4,2021-06-05 13:53:28.377359,Books,Flipcart,494.128492
...,...,...,...,...,...
95,95,2021-07-19 13:53:28.377359,Utility,Phone,388.671213
96,96,2021-01-12 19:53:28.377359,Books,Flipcart,467.554562
97,97,2021-03-25 11:53:28.377359,Utility,Phone,320.789434
98,98,2021-05-13 15:53:28.377359,Travel,Taxi,442.096469


In [67]:
!ls upload.pdf

upload.pdf


## Regular expressions


Regular are used to search patterns in text!

In [68]:
import re

In [71]:
empty_pattern = re.compile("^$") # empty line ... ^ means start of line, $ means end of line
lines_with_sigle_char = re.compile("^.$") # . means any char
date_like_pattern = re.compile(r"^\d\d\d\d-\d{1,2}-\d{1,2}") # \d is for digit, 
                                                            #{x, y} whatever is before this will come min x times and max y times

In [72]:
lines="""line1
1
2
3
ksjdksa dfjkh fdgjkh


2026-2-14
2025-12-12
sadas



fsdfsf"""

In [73]:
def matchpatterns(lines, p):
    for line in lines.split("\n"):
        if p.match(line):
            print("Found a patterns!")
            print(line)

In [74]:
matchpatterns(lines, empty_pattern)

Found a patterns!

Found a patterns!

Found a patterns!

Found a patterns!

Found a patterns!



In [75]:
matchpatterns(lines, lines_with_sigle_char)

Found a patterns!
1
Found a patterns!
2
Found a patterns!
3


In [76]:
matchpatterns(lines, date_like_pattern)

Found a patterns!
2026-2-14
Found a patterns!
2025-12-12


## activating virtual env through .bat file

In [78]:
%%file jupyler-launch.bat
call c:\usrs\vikrant\trainingenv\Scripts\activate.bat
jupyterlab

Overwriting jupyler-launch.bat
